# Error Functions in Machine Learning

An **error function** (also called a loss or cost function) tells us how far off our model's predictions are from the true answers. The goal of training is to **minimize** this error.

Think of it like a grade on a test — the error function scores how badly the model performed, and the model learns by trying to lower that score.

## Binary Classification Error

$$E = -\frac{1}{m} \sum_{i=1}^{m} (1 - y_i) \ln(1 - \hat{y}_i) + y_i \ln(\hat{y}_i)$$

This is the **cross-entropy loss** for binary classification (2 classes: 0 or 1).

| Symbol | Meaning |
|--------|---------|
| $m$ | Total number of training samples |
| $y_i$ | True label for sample $i$ (either 0 or 1) |
| $\hat{y}_i$ | Model's predicted probability that sample $i$ is class 1 |
| $\frac{1}{m}$ | Averages the error over all samples |

**How it works:** For each sample, only one term is "active":
- If $y_i = 1$: we compute $\ln(\hat{y}_i)$ → penalizes predictions far from 1
- If $y_i = 0$: we compute $\ln(1 - \hat{y}_i)$ → penalizes predictions far from 0

> **Real-life example:** Imagine a spam filter classifying emails. For an email that IS spam ($y = 1$), if the model predicts 0.95 probability of spam, the penalty is $-\ln(0.95) \approx 0.05$ (small — good job). But if it predicts 0.1, the penalty is $-\ln(0.1) \approx 2.3$ (huge — it was very wrong and very confident about it). The error function punishes confident wrong answers much harder than uncertain ones.

## Total Error as a Function of Weights and Bias

$$E(W, b) = -\frac{1}{m} \sum_{i=1}^{m} (1 - y_i) \ln(1 - \sigma(Wx^{(i)} + b)) + y_i \ln(\sigma(Wx^{(i)} + b))$$

This is the same error function, but written to show **what the model actually computes**:

| Symbol | Meaning |
|--------|---------|
| $W$ | Weights — how much each input feature matters |
| $b$ | Bias — a constant offset |
| $x^{(i)}$ | The input features for sample $i$ |
| $\sigma(\cdot)$ | The sigmoid function — squishes any number into the range (0, 1) |

The flow is: `inputs → linear combination (Wx + b) → sigmoid → probability → error`

This form is useful because **gradient descent** needs to know how the error depends on $W$ and $b$ to update them.

> **Real-life example:** Think of a loan approval model. The inputs $x$ might be income and credit score. The weights $W$ decide how important each factor is (maybe credit score matters more than income). The bias $b$ sets a baseline tendency (approve or deny by default). The sigmoid squishes the result into a probability like "78% chance of approval." This formula shows us exactly where to look when the model gets it wrong — we can trace the error back to specific weights and tweak them.

## Multiclass Error

$$E = -\frac{1}{m} \sum_{i=1}^{m} \sum_{j=1}^{n} y_{ij} \ln(\hat{y}_{ij})$$

When there are more than 2 classes (e.g., recognizing digits 0–9):

| Symbol | Meaning |
|--------|---------|
| $n$ | Number of classes |
| $y_{ij}$ | 1 if sample $i$ belongs to class $j$, else 0 (one-hot encoded) |
| $\hat{y}_{ij}$ | Model's predicted probability that sample $i$ is class $j$ |

The double sum goes over every sample AND every class. For each sample, only the correct class contributes (since $y_{ij} = 0$ for wrong classes).

> **Real-life example:** A model that recognizes handwritten digits (0–9). You show it the number "7" and it outputs probabilities: `[0.01, 0.02, 0.01, 0.05, 0.01, 0.02, 0.03, 0.80, 0.03, 0.02]`. Only the 7th position matters (the true class), so the error is $-\ln(0.80) \approx 0.22$. If it had predicted 0.99 for "7", the error would be just $-\ln(0.99) \approx 0.01$. The model doesn't get penalized for what it predicted on the wrong classes — only for how confident it was about the right one.

## Derivative of the Sigmoid Function

$$\sigma'(x) = \sigma(x)(1 - \sigma(x))$$

This is the **derivative** of the sigmoid function, which is essential for **backpropagation** — the algorithm used to update weights during training.

| Symbol | Meaning |
|--------|---------|
| $\sigma(x)$ | The sigmoid function: $\sigma(x) = \frac{1}{1 + e^{-x}}$ |
| $\sigma'(x)$ | The rate of change of the sigmoid at point $x$ |

**Why it matters:** During gradient descent, we need to compute how the error changes with respect to the weights. Since the sigmoid is part of the model, its derivative appears in the chain rule calculations.

**Key property:** The derivative is expressed entirely in terms of the sigmoid itself, making it computationally efficient — no need to recalculate from scratch.

**Note:** The maximum value of $\sigma'(x)$ is $0.25$, occurring at $x = 0$. This means gradients can become very small for large $|x|$, which is known as the **vanishing gradient problem**.

> **Real-life example:** Imagine a dimmer switch for a light. The sigmoid is the switch — it smoothly transitions from off (0) to on (1). The derivative tells us how *sensitive* the switch is at any position. In the middle ($x = 0$), a tiny push changes the brightness noticeably ($\sigma' = 0.25$). But near the extremes (fully on or fully off), you can push hard and the brightness barely changes ($\sigma' \approx 0$). This is why deep networks struggle to learn through many layers — the "signal" gets weaker each time it passes through a sigmoid that's near its extremes.

## Gradient of the Error Function

$$\nabla E = \left( \frac{\partial E}{\partial w_1}, \cdots, \frac{\partial E}{\partial w_n}, \frac{\partial E}{\partial b} \right)$$

The **gradient** is a vector of all partial derivatives of the error function with respect to each parameter. It points in the direction of **steepest increase** of the error.

| Symbol | Meaning |
|--------|---------|
| $\nabla E$ | The gradient vector of the error function |
| $\frac{\partial E}{\partial w_j}$ | How much the error changes when weight $w_j$ changes slightly |
| $\frac{\partial E}{\partial b}$ | How much the error changes when the bias $b$ changes slightly |
| $n$ | Number of input features (and corresponding weights) |

**Why it matters:** In **gradient descent**, we update the weights and bias by moving in the **opposite direction** of the gradient to minimize the error:

$$w_j \leftarrow w_j - \alpha \frac{\partial E}{\partial w_j}, \quad b \leftarrow b - \alpha \frac{\partial E}{\partial b}$$

where $\alpha$ is the **learning rate** — a hyperparameter that controls the step size.

**Key insight:** The gradient tells the model *which direction* to adjust each parameter and *by how much*, making it the core mechanism behind learning.

> **Real-life example:** You're standing on a foggy hill and want to reach the lowest point (the valley). You can't see the valley, but you can feel the slope under your feet. The gradient is like feeling the slope in *every* direction at once and knowing exactly which way is steepest downhill. Each partial derivative is like asking "if I take one step in this specific direction (adjust one weight), does the ground go up or down, and by how much?" You then take a step in the overall steepest-downhill direction. The learning rate $\alpha$ is how big your step is — too big and you overshoot the valley, too small and you'll take forever to get there.

## Gradient Calculation — Step-by-Step Derivation

Now that we know the gradient is the vector of partial derivatives, let's actually **compute** it step by step.

To simplify the calculation, we focus on the error produced by a **single point** first. The total error is just the average of all individual errors, so once we know the derivative for one point, we can average over all $m$ points.

The error produced by a single point is:

$$E = -y \ln(\hat{y}) - (1 - y) \ln(1 - \hat{y})$$

where the prediction is $\hat{y} = \sigma(Wx + b)$.

Our goal is to compute the partial derivatives $\frac{\partial E}{\partial w_j}$ and $\frac{\partial E}{\partial b}$.

> **Observation:** We focus on one sample first because calculus works the same way on each sample individually. Since the total error is just the average, we can derive the formula for one point and it applies to all of them. It's like figuring out the recipe for one cookie — once you know it, making a batch is just repetition.

### Step 1: Derivative of $\hat{y}$ with respect to $w_j$

Since $\hat{y} = \sigma(Wx + b)$, we apply the **chain rule**. Recall that $\sigma'(x) = \sigma(x)(1 - \sigma(x))$, so:

$$\frac{\partial}{\partial w_j} \hat{y} = \frac{\partial}{\partial w_j} \sigma(Wx + b) = \sigma'(Wx + b) \cdot \frac{\partial}{\partial w_j}(Wx + b)$$

Now, expanding $Wx + b = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b$, the only term that depends on $w_j$ is $w_j x_j$, so:

$$\frac{\partial}{\partial w_j}(Wx + b) = x_j$$

Putting it together:

$$\frac{\partial \hat{y}}{\partial w_j} = \sigma'(Wx + b) \cdot x_j = \hat{y}(1 - \hat{y}) \cdot x_j$$

> **Observation:** This result tells us that the sensitivity of the prediction to a weight change depends on two things: how "active" the sigmoid is at that point ($\hat{y}(1 - \hat{y})$), and how large the corresponding input is ($x_j$). If an input feature is zero, changing its weight has no effect on the prediction. Think of a salary prediction model — if someone's "years of experience" is 0, then the weight for "years of experience" doesn't affect that person's prediction at all.

### Step 2: Derivative of $E$ with respect to $w_j$

Now we compute $\frac{\partial E}{\partial w_j}$ using the chain rule on $E = -y \ln(\hat{y}) - (1 - y) \ln(1 - \hat{y})$:

$$\frac{\partial E}{\partial w_j} = -y \cdot \frac{1}{\hat{y}} \cdot \frac{\partial \hat{y}}{\partial w_j} - (1 - y) \cdot \frac{1}{1 - \hat{y}} \cdot (-1) \cdot \frac{\partial \hat{y}}{\partial w_j}$$

Substituting $\frac{\partial \hat{y}}{\partial w_j} = \hat{y}(1 - \hat{y}) \cdot x_j$ from Step 1:

$$= -y \cdot \frac{1}{\hat{y}} \cdot \hat{y}(1 - \hat{y}) \cdot x_j + (1 - y) \cdot \frac{1}{1 - \hat{y}} \cdot \hat{y}(1 - \hat{y}) \cdot x_j$$

$$= -y(1 - \hat{y}) \cdot x_j + (1 - y)\hat{y} \cdot x_j$$

$$= \left[-y + y\hat{y} + \hat{y} - y\hat{y}\right] x_j$$

$$= (\hat{y} - y) \cdot x_j = -(y - \hat{y}) \cdot x_j$$

**Result:**

$$\frac{\partial E}{\partial w_j} = -(y - \hat{y}) \cdot x_j$$

> **Observation:** Notice how all the sigmoid complexity cancels out and we end up with something beautifully simple: the gradient is just (prediction error) × (input). This is not a coincidence — it's a property of the cross-entropy loss paired with the sigmoid. Think of a teacher grading a student's answer: the "correction" is proportional to how wrong the answer was, and it's applied to the specific knowledge (input) that led to the mistake.

### Step 3: Derivative of $E$ with respect to $b$

A similar calculation (replacing $w_j$ with $b$) gives us:

$$\frac{\partial E}{\partial b} = -(y - \hat{y})$$

This follows the same logic as Step 2, except that $\frac{\partial}{\partial b}(Wx + b) = 1$ instead of $x_j$.

> **Observation:** The bias gradient is just the raw prediction error without any input multiplied in. This makes sense — the bias shifts the decision boundary regardless of the input values. It's like the "default attitude" of the model. If the model is consistently too pessimistic (predicting too low), the bias correction will push it up across the board, no matter what the inputs are.

### Step 4: The Full Gradient — A Beautiful Result

Combining the results from Steps 2 and 3, for a point with coordinates $(x_1, \ldots, x_n)$, label $y$, and prediction $\hat{y}$, the gradient of the error function at that point is:

$$\nabla E = \left(-(y - \hat{y})x_1, \;\cdots, \;-(y - \hat{y})x_n, \;-(y - \hat{y})\right)$$

This can be written compactly as:

$$\nabla E = -(y - \hat{y})(x_1, \ldots, x_n, 1)$$

**Why this is fascinating:** The gradient is a **scalar** times the coordinates of the point (with a 1 appended for the bias). And what is that scalar? It's $-(y - \hat{y})$ — a multiple of the **difference between the label and the prediction**.

| Scenario | $(y - \hat{y})$ | Effect |
|----------|-----------------|--------|
| Label is 1, prediction close to 0 | Large positive | Large gradient → big update toward the point |
| Label is 0, prediction close to 1 | Large negative | Large gradient → big update away from the point |
| Prediction is close to label | Near zero | Small gradient → tiny update (already doing well) |

**Intuition:** A misclassified point produces a large gradient, pushing the model to correct its mistake. A correctly classified point barely affects the weights. This is exactly the behavior we want from a learning algorithm.

> **Real-life example:** Imagine a hiring model that predicts whether a candidate will succeed. Candidate A was predicted 90% success and indeed succeeded — the model barely adjusts (it was right). Candidate B was predicted 10% success but actually excelled — the model makes a huge correction, effectively saying "pay more attention to people like B." The gradient automatically focuses the model's learning effort on the examples it got most wrong.

### Gradient Descent Step

So, a small gradient means we'll change our coordinates by a little bit, and a large gradient means we'll change our coordinates by a lot. If this sounds anything like the perceptron algorithm, this is no coincidence!

Since the gradient descent step consists in **subtracting a multiple of the gradient** of the error function at every point, the weights update as:

$$w_i' \leftarrow w_i - \alpha \left[-(y - \hat{y})x_i\right]$$

which simplifies to:

$$w_i' \leftarrow w_i + \alpha (y - \hat{y}) x_i$$

Similarly, the bias updates as:

$$b' \leftarrow b + \alpha (y - \hat{y})$$

| Symbol | Meaning |
|--------|---------|
| $w_i$ | Current weight for feature $i$ |
| $w_i'$ | Updated weight for feature $i$ after one gradient descent step |
| $\alpha$ | Learning rate — controls the step size of each update |
| $y$ | True label for the sample (0 or 1) |
| $\hat{y}$ | Model's predicted probability for the sample |
| $x_i$ | The $i$-th input feature of the sample |
| $b$ | Current bias |
| $b'$ | Updated bias after one gradient descent step |

**Note:** Since we've taken the average of the errors, the term we are adding should technically be $\frac{1}{m} \cdot \alpha$ instead of $\alpha$. But since $\alpha$ is a constant, we simplify by absorbing $\frac{1}{m}$ into the learning rate and just call the whole thing $\alpha$.

> **Real-life example:** Think of tuning a recipe. You made a cake that's too dry ($y - \hat{y}$ is large). The learning rate $\alpha$ is how bold you are with changes — a cautious baker might add a tiny bit more butter (small $\alpha$), while an aggressive one might double it (large $\alpha$). The input $x_i$ determines *which* ingredient gets adjusted — if you used a lot of flour ($x_i$ is large), the flour weight gets a bigger correction because flour had a bigger role in making it dry. Each baking attempt (training step) gets you closer to the perfect recipe.

### Logistic Regression Algorithm

Here are our steps for logistic regression:

1. Start with random weights: $w_1, \ldots, w_n, b$
2. For every point $(x_1, \ldots, x_n)$:
    - For $i = 1 \ldots n$:
      - Update $w_i' \leftarrow w_i + \alpha (y - \hat{y}) x_i$
    - Update $b' \leftarrow b + \alpha (y - \hat{y})$
3. Repeat until the error is small

> **Observation:** This is the complete learning algorithm — surprisingly simple given everything we derived. The model starts knowing nothing (random weights), looks at each example, computes how wrong it was, and nudges the weights in the right direction. After enough iterations, the weights converge to values that draw the best possible decision boundary. It's like learning to ride a bike — each wobble (error) teaches you to adjust, and eventually you balance without thinking.

### Gradient Descent vs Perceptron Algorithm

**Gradient Descent**

With gradient descent, we change the weights from $w_i$ to $w_i + \alpha (y - \hat{y}) x_i$.

**Perceptron Algorithm**

With the perceptron algorithm we only change the weights on the misclassified points. If a point $x$ is misclassified:

We change $w_i$:

- To $w_i + \alpha x_i$ if positive
- To $w_i - \alpha x_i$ if negative

If correctly classified: $y - \hat{y} = 0$

If misclassified:

- $y - \hat{y} = 1$ if positive
- $y - \hat{y} = -1$ if negative

> **Observation:** The perceptron is "all or nothing" — it only cares about misclassified points and always applies the same correction magnitude. Gradient descent is smoother — it adjusts for *every* point, with corrections proportional to *how wrong* the prediction was. Think of two basketball coaches: the perceptron coach only gives feedback when you miss the shot entirely ("try harder!"). The gradient descent coach gives nuanced feedback on every shot ("you were 2 inches to the left, adjust slightly"). Both eventually improve the player, but gradient descent converges more smoothly because it uses richer information.

## Feedforward

Feedforward is the process neural networks use to turn an input into an output. It works in three steps:

1. Take the input vector
2. Apply a sequence of linear transformations and sigmoid activations
3. Produce the final prediction

> **Observation:** Feedforward is just the model making a prediction — it's the "thinking" phase. No learning happens here, just computation. It's like a factory assembly line: raw materials (inputs) go in, pass through stations (layers) where workers (weights) transform them, and a finished product (prediction) comes out the other end. Every time you ask a trained model "is this a cat?", it runs feedforward.

### Feedforward Formula (one hidden layer)

$$\hat{y} = \sigma\left(W^{(2)} \cdot \sigma\left(W^{(1)} x + b^{(1)}\right) + b^{(2)}\right)$$

| Symbol | Meaning |
|--------|---------|
| $x = \begin{pmatrix} x_1 \\ x_2 \end{pmatrix}$ | Input vector (2 features) |
| $W^{(1)} = \begin{pmatrix} W_{11}^{(1)} & W_{21}^{(1)} \\ W_{12}^{(1)} & W_{22}^{(1)} \end{pmatrix}$ | Weights from input to hidden layer |
| $b^{(1)} = \begin{pmatrix} b_1^{(1)} \\ b_2^{(1)} \end{pmatrix}$ | Biases for the hidden layer |
| $W^{(2)} = \begin{pmatrix} W_{11}^{(2)} & W_{21}^{(2)} \end{pmatrix}$ | Weights from hidden layer to output |
| $b^{(2)}$ | Bias for the output layer |
| $\sigma$ | Sigmoid activation function |

**How it flows:** input $x$ → multiply by $W^{(1)}$, add $b^{(1)}$ → apply $\sigma$ → multiply by $W^{(2)}$, add $b^{(2)}$ → apply $\sigma$ → prediction $\hat{y}$

> **Real-life example:** Suppose you're predicting whether someone will buy a product based on age ($x_1$) and income ($x_2$). The hidden layer creates *new features* from the originals — one hidden neuron might learn to detect "young and wealthy" while another detects "older with moderate income." The output layer then combines these higher-level features into the final yes/no prediction. A single linear model can only draw a straight line between buyers and non-buyers; the hidden layer lets it draw curves.

#### Expanded — with all vectors written out

$$\hat{y} = \sigma\left(\begin{pmatrix} W_{11}^{(2)} & W_{21}^{(2)} \end{pmatrix} \cdot \sigma\left(\begin{pmatrix} W_{11}^{(1)} & W_{21}^{(1)} \\ W_{12}^{(1)} & W_{22}^{(1)} \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} + \begin{pmatrix} b_1^{(1)} \\ b_2^{(1)} \end{pmatrix}\right) + b^{(2)}\right)$$

Reading inside-out:

1. $\begin{pmatrix} W_{11}^{(1)} & W_{21}^{(1)} \\ W_{12}^{(1)} & W_{22}^{(1)} \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} + \begin{pmatrix} b_1^{(1)} \\ b_2^{(1)} \end{pmatrix}$ — linear combination in the hidden layer (produces a 2-element vector)

2. $\sigma(\cdots)$ — apply sigmoid to each element of that vector

3. $\begin{pmatrix} W_{11}^{(2)} & W_{21}^{(2)} \end{pmatrix} \cdot (\text{hidden output}) + b^{(2)}$ — linear combination in the output layer (produces a scalar)

4. $\sigma(\cdots)$ — apply sigmoid to get the final prediction $\hat{y}$

### Multi-Layer Perceptron (multiple hidden layers)

$$\hat{y} = \sigma\Big(W^{(3)} \cdot \sigma\Big(W^{(2)} \cdot \sigma\Big(W^{(1)} x\Big)\Big)\Big)$$

This is the same idea extended to deeper networks — just stack more layers of (linear transformation → sigmoid) on top of each other. Each layer combines the previous layer's outputs in new ways, allowing the network to learn increasingly complex patterns.

> **Real-life example:** Think of image recognition. Layer 1 detects simple things: edges and color changes. Layer 2 combines edges into shapes: circles, rectangles, curves. Layer 3 combines shapes into objects: "that's an eye," "that's a wheel." Each layer builds on the previous one to understand increasingly abstract concepts. A single layer can only see pixels — it takes depth to see meaning.

#### Expanded — with biases included

$$\hat{y} = \sigma\Big(W^{(3)} \cdot \sigma\Big(W^{(2)} \cdot \sigma\Big(W^{(1)} x + b^{(1)}\Big) + b^{(2)}\Big) + b^{(3)}\Big)$$

Reading inside-out:

1. $W^{(1)} x + b^{(1)}$ — linear transformation at layer 1
2. $\sigma(\cdots)$ — activate layer 1 output
3. $W^{(2)} \cdot (\text{layer 1 output}) + b^{(2)}$ — linear transformation at layer 2
4. $\sigma(\cdots)$ — activate layer 2 output
5. $W^{(3)} \cdot (\text{layer 2 output}) + b^{(3)}$ — linear transformation at output layer
6. $\sigma(\cdots)$ — final activation → prediction $\hat{y}$

Each layer has its own weight matrix $W^{(k)}$ and bias vector $b^{(k)}$, so the total number of learnable parameters grows with each added layer.

#### Expanded — with all vectors written out

For a network with 2 inputs, 3 hidden neurons in layer 1, 2 hidden neurons in layer 2, and 1 output:

$$\hat{y} = \sigma\left(\begin{pmatrix} W_{11}^{(3)} & W_{21}^{(3)} \end{pmatrix} \cdot \sigma\left(\begin{pmatrix} W_{11}^{(2)} & W_{21}^{(2)} & W_{31}^{(2)} \\ W_{12}^{(2)} & W_{22}^{(2)} & W_{32}^{(2)} \end{pmatrix} \cdot \sigma\left(\begin{pmatrix} W_{11}^{(1)} & W_{21}^{(1)} \\ W_{12}^{(1)} & W_{22}^{(1)} \\ W_{13}^{(1)} & W_{23}^{(1)} \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} + \begin{pmatrix} b_1^{(1)} \\ b_2^{(1)} \\ b_3^{(1)} \end{pmatrix}\right) + \begin{pmatrix} b_1^{(2)} \\ b_2^{(2)} \end{pmatrix}\right) + b^{(3)}\right)$$

Reading inside-out:

1. $\begin{pmatrix} W_{11}^{(1)} & W_{21}^{(1)} \\ W_{12}^{(1)} & W_{22}^{(1)} \\ W_{13}^{(1)} & W_{23}^{(1)} \end{pmatrix} \begin{pmatrix} x_1 \\ x_2 \end{pmatrix} + \begin{pmatrix} b_1^{(1)} \\ b_2^{(1)} \\ b_3^{(1)} \end{pmatrix}$ — layer 1: 2 inputs → 3 outputs

2. $\sigma(\cdots)$ — activate each of the 3 outputs

3. $\begin{pmatrix} W_{11}^{(2)} & W_{21}^{(2)} & W_{31}^{(2)} \\ W_{12}^{(2)} & W_{22}^{(2)} & W_{32}^{(2)} \end{pmatrix} \cdot (\text{layer 1 output}) + \begin{pmatrix} b_1^{(2)} \\ b_2^{(2)} \end{pmatrix}$ — layer 2: 3 inputs → 2 outputs

4. $\sigma(\cdots)$ — activate each of the 2 outputs

5. $\begin{pmatrix} W_{11}^{(3)} & W_{21}^{(3)} \end{pmatrix} \cdot (\text{layer 2 output}) + b^{(3)}$ — output layer: 2 inputs → 1 scalar

6. $\sigma(\cdots)$ — final activation → prediction $\hat{y}$

**Notice the pattern:** the matrix dimensions match the connections — $W^{(k)}$ has shape (neurons in layer $k$) × (neurons in layer $k-1$).

## Backpropagation

Backpropagation is the algorithm that computes the gradient of the error function with respect to **every weight in the network** — across all layers.

The key idea: we already know how to compute the gradient for the output layer (same as logistic regression). The challenge is computing the gradient for the **hidden layers**, where weights don't directly produce the prediction. Backpropagation solves this by propagating the error **backwards** from the output, layer by layer.

**Why "back"?** In feedforward, data flows forward: input → hidden → output. In backpropagation, gradients flow backward: output → hidden → input.

> **Real-life example:** Imagine a relay race where the team loses. The coach needs to figure out who caused the loss. It's easy to evaluate the last runner (output layer) — you can compare their time directly. But to evaluate the first runner (hidden layer), you need to trace the impact backwards: "the second runner started late *because* the first runner's handoff was slow." Backpropagation does exactly this — it traces the blame for the error backwards through each layer to figure out how every weight contributed.

### The Chain Rule for Multi-Layer Networks

For a single-layer model, we computed:

$$\frac{\partial E}{\partial w_j} = \frac{\partial E}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial w_j}$$

For a multi-layer network, the chain rule extends. Consider a 2-layer network where:
- Layer 1 produces hidden outputs: $h = \sigma(W^{(1)} x + b^{(1)})$
- Layer 2 produces the prediction: $\hat{y} = \sigma(W^{(2)} h + b^{(2)})$

To find how the error depends on a weight in **layer 2** ($W^{(2)}$), the chain is short:

$$\frac{\partial E}{\partial W^{(2)}} = \frac{\partial E}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial W^{(2)}}$$

To find how the error depends on a weight in **layer 1** ($W^{(1)}$), the chain is longer — it must pass through layer 2:

$$\frac{\partial E}{\partial W^{(1)}} = \frac{\partial E}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial h} \cdot \frac{\partial h}{\partial W^{(1)}}$$

| Symbol | Meaning |
|--------|---------|
| $h$ | Output of the hidden layer (a vector) |
| $\frac{\partial E}{\partial \hat{y}}$ | How the error changes with the prediction (we already know this) |
| $\frac{\partial \hat{y}}{\partial h}$ | How the prediction changes with the hidden outputs |
| $\frac{\partial h}{\partial W^{(1)}}$ | How the hidden outputs change with layer 1 weights |

**Key insight:** Each link in the chain is a derivative we can compute. The chain rule lets us multiply them together to "reach back" to any layer.

> **Real-life example:** Imagine a company where the CEO (output) makes decisions based on reports from managers (hidden layer), who base their reports on data from analysts (input layer). If the company makes a bad decision, tracing the blame means asking: "How much did the CEO's interpretation change the outcome?" × "How much did each manager's report influence the CEO?" × "How much did each analyst's work influence their manager?" The chain rule multiplies these influences together to figure out each person's responsibility — even those far from the final decision.

### Step 1: Gradient at the Output Layer

We define the **error signal** at the output as $\delta^{(2)}$ — how much the output node is "responsible" for the error.

For the output layer, this is the same result we derived earlier for logistic regression:

$$\delta^{(2)} = \hat{y} - y$$

Then the gradients for the output layer weights and bias are:

$$\frac{\partial E}{\partial W_{j}^{(2)}} = \delta^{(2)} \cdot h_j$$

$$\frac{\partial E}{\partial b^{(2)}} = \delta^{(2)}$$

| Symbol | Meaning |
|--------|---------|
| $\delta^{(2)}$ | Error signal at the output layer |
| $\hat{y}$ | Model's prediction (output of the last sigmoid) |
| $y$ | True label |
| $h_j$ | The $j$-th output of the hidden layer (input to this layer) |

**This is familiar:** the gradient is the error signal times the input to that layer — same pattern as the single-layer case, just using hidden outputs $h_j$ instead of raw inputs $x_j$.

> **Observation:** The output layer gradient is the "easy" part of backpropagation — it's identical to what we derived for logistic regression. This is because the output layer directly produces the prediction, so there's no extra chain to follow. It's like grading the last student to touch a group project — their contribution is directly visible in the final result.

### Step 2: Gradient at the Hidden Layer — The "Back" Part

Now the new piece: to update $W^{(1)}$, we need to know how much each hidden neuron contributed to the error. We compute the **error signal** for each hidden neuron $j$:

$$\delta_j^{(1)} = \left(\sum_{k} W_{jk}^{(2)} \cdot \delta_k^{(2)}\right) \cdot h_j(1 - h_j)$$

For our 2-layer network with one output, this simplifies to:

$$\delta_j^{(1)} = W_{j}^{(2)} \cdot \delta^{(2)} \cdot h_j(1 - h_j)$$

| Part | What it does |
|------|-------------|
| $W_{j}^{(2)} \cdot \delta^{(2)}$ | Takes the output error and weights it by how much hidden neuron $j$ contributed (via $W_j^{(2)}$) |
| $h_j(1 - h_j)$ | The sigmoid derivative at hidden neuron $j$ — scales the error by how sensitive the sigmoid was at that point |

Then the gradients for the hidden layer weights and bias are:

$$\frac{\partial E}{\partial W_{ij}^{(1)}} = \delta_j^{(1)} \cdot x_i$$

$$\frac{\partial E}{\partial b_j^{(1)}} = \delta_j^{(1)}$$

**Same pattern again:** the gradient is the error signal times the input to that layer. The only difference is how $\delta$ is computed — at the output it's $(\hat{y} - y)$, at hidden layers it's propagated back from the layer above.

> **Real-life example:** Imagine a restaurant where food goes through a sous chef (hidden layer) before the head chef (output) serves it. If a dish is too salty ($\delta^{(2)}$), how much blame does the sous chef get? It depends on two things: (1) how much the head chef relied on their preparation ($W_j^{(2)}$ — if the head chef barely used their sauce, the sous chef isn't very responsible), and (2) how much room the sous chef had to change things ($h_j(1 - h_j)$ — if they were already at the extreme of adding maximum or zero salt, they couldn't have done much differently). The hidden layer error signal captures both factors.

### Step 3: The General Pattern

For **any** layer $l$ in a deep network, the error signal follows the same recursive formula:

$$\delta^{(l)} = \left({W^{(l+1)}}^T \cdot \delta^{(l+1)}\right) \odot h^{(l)} \odot (1 - h^{(l)})$$

| Symbol | Meaning |
|--------|---------|
| $\delta^{(l)}$ | Error signal at layer $l$ (a vector with one element per neuron) |
| ${W^{(l+1)}}^T$ | Transpose of the weight matrix connecting layer $l$ to layer $l+1$ |
| $\delta^{(l+1)}$ | Error signal from the layer above (already computed) |
| $\odot$ | Element-wise multiplication (Hadamard product) |
| $h^{(l)}$ | Output of layer $l$ (after sigmoid activation) |

And the gradients at layer $l$ are:

$$\frac{\partial E}{\partial W^{(l)}} = \delta^{(l)} \cdot {h^{(l-1)}}^T$$

$$\frac{\partial E}{\partial b^{(l)}} = \delta^{(l)}$$

Where $h^{(l-1)}$ is the output of the previous layer (or the input $x$ if $l = 1$).

**The pattern at every layer is always:**
1. Get the error signal from the layer above
2. Weight it by how much this layer contributed ($W^T$)
3. Scale by the sigmoid derivative ($h(1-h)$)
4. Multiply by the input to get the weight gradient

> **Observation:** This recursive pattern is what makes backpropagation so elegant. Instead of deriving the gradient from scratch for every layer — which would be incredibly tedious for a 100-layer network — you just pass the error signal backwards one layer at a time, reusing the same formula. It's like a chain of dominoes: you only need to push the first one (compute $\delta$ at the output), and each domino knocks over the next one in the same predictable way. This is why training deep networks is computationally *feasible* — the work scales linearly with the number of layers, not exponentially.

### The Backpropagation Algorithm

Putting feedforward and backpropagation together, here is the full training loop:

**1. Feedforward** — compute all outputs, layer by layer:

$$z^{(k)} = W^{(k)} h^{(k-1)} + b^{(k)}, \quad h^{(k)} = \sigma(z^{(k)})$$

**2. Output error** — compare prediction with true label:

$$\delta^{(L)} = \hat{y} - y$$

**3. Backpropagate** — compute error signals from last hidden layer back to first:

$$\delta^{(k)} = \left({W^{(k+1)}}^T \cdot \delta^{(k+1)}\right) \odot h^{(k)} \odot (1 - h^{(k)})$$

**4. Update** — adjust all weights and biases using the error signals:

$$W^{(k)} \leftarrow W^{(k)} - \alpha \cdot \delta^{(k)} \cdot {h^{(k-1)}}^T$$

$$b^{(k)} \leftarrow b^{(k)} - \alpha \cdot \delta^{(k)}$$

**5. Repeat** — go back to step 1 with the next training sample (or batch)

> **Real-life example:** Training a neural network is like learning to play darts blindfolded. Each round: (1) **Throw** the dart — feedforward produces a prediction. (2) **Someone tells you** how far off you were — the output error $\delta^{(L)}$. (3) **You mentally trace** what went wrong: "My arm angle was off because my grip was off because my stance was off" — backpropagation traces the error through each layer. (4) **You adjust** each part of your form by a small amount ($\alpha$) — weight update. (5) **Throw again.** After thousands of throws, you consistently hit near the bullseye — the model converges to low error.

---

**The beautiful symmetry:**

| Direction | What flows | Algorithm |
|-----------|-----------|-----------|
| Forward (input → output) | Data: $x \to h^{(1)} \to h^{(2)} \to \cdots \to \hat{y}$ | Feedforward |
| Backward (output → input) | Errors: $\delta^{(L)} \to \delta^{(L-1)} \to \cdots \to \delta^{(1)}$ | Backpropagation |

The network learns by alternating these two passes: forward to make a prediction, backward to figure out how to improve it.

> **Observation:** This 5-step loop is the engine behind virtually every neural network ever trained — from simple classifiers to GPT and image generators. The math is the same whether you have 2 layers or 200. What changes is the scale (more layers, more neurons, more data), not the fundamental algorithm. Once you understand this loop, you understand the heart of deep learning.